# Week 7, days 2-4 -- QLoRA fine-tune

**Run this in Colab on a T4 (free) or an A100.** Nothing here works on a laptop: it needs a CUDA GPU
for 4-bit quantisation.

The plan: load a 3B base model in 4-bit, attach LoRA adapters to the attention projections, and train
on the tasting-note prompts so the model completes `Price is $` with a number. Only the adapters
train -- about 0.5% of the parameters -- which is what makes this fit in 16GB.

In [ ]:
!pip install -q "transformers>=4.44" "peft>=0.13" "trl>=0.11" "bitsandbytes>=0.44" \
    "datasets>=3.0" "accelerate>=1.0"
!git clone -q https://github.com/borjahernandez/wine-pricer.git
%cd wine-pricer

In [ ]:
import torch
from datasets import load_dataset
from google.colab import userdata
from huggingface_hub import login
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import DataCollatorForCompletionOnlyLM, SFTConfig, SFTTrainer

login(userdata.get("HF_TOKEN"))

BASE_MODEL = "Qwen/Qwen2.5-3B"
DATASET = "borjahernandez/wine-pricer"
RUN = "wine-pricer-qwen3b"
PREFIX = "Price is $"  # the response template: loss is computed on what follows it

### Hyperparameters

Sensible starting points, all worth a sweep:

| knob | value | why |
| --- | --- | --- |
| `r` | 32 | adapter rank. 8 underfits here, 64 costs memory for little gain |
| `alpha` | 64 | conventionally 2r |
| target modules | attention projections | where the task-specific reasoning lives |
| `lr` | 1e-4 | LoRA tolerates rates ~10x a full fine-tune |
| epochs | 1 | 50k examples is plenty; a second epoch mostly memorises |
| 4-bit nf4, double quant | on | the whole reason this fits on a T4 |

In [ ]:
LORA = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

QUANT = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

CONFIG = SFTConfig(
    output_dir=RUN,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # effective batch 16
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="paged_adamw_32bit",
    max_seq_length=256,
    dataset_text_field="prompt",
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    bf16=True,
    report_to="none",
    push_to_hub=True,
    hub_model_id=f"borjahernandez/{RUN}",
    hub_private_repo=True,
)

In [ ]:
data = load_dataset(DATASET)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=QUANT, device_map="auto")
model.generation_config.pad_token_id = tokenizer.pad_token_id

# Train on the answer only: without this the model spends its capacity learning to recite tasting notes.
collator = DataCollatorForCompletionOnlyLM(response_template=PREFIX, tokenizer=tokenizer)

trainer = SFTTrainer(
    model=model,
    train_dataset=data["train"],
    peft_config=LORA,
    args=CONFIG,
    data_collator=collator,
)
trainer.train()
trainer.push_to_hub(f"Fine-tuned on {DATASET}")

### Score it on the same test split as everything else

Two ways to read the answer out:

1. **Generate** a few tokens and parse the number.
2. **Weighted average over the logits** of the first answer token -- the model's whole distribution
   instead of its argmax, which is measurably better calibrated for a numeric target.

Both go through `pricer.evaluator`, so the result drops straight onto the week-6 leaderboard.

In [ ]:
import re

from pricer.evaluator import evaluate
from pricer.items import Wine

_, _, test = Wine.from_hub(DATASET)
model.eval()


def parse_price(text: str) -> float:
    match = re.search(r"[-+]?\d[\d,]*\.?\d*", text.replace("$", ""))
    return float(match.group().replace(",", "")) if match else 0.0


def specialist(wine: Wine) -> float:
    inputs = tokenizer(wine.test_prompt(), return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=6, do_sample=False)
    completion = tokenizer.decode(output[0][inputs["input_ids"].shape[1] :])
    return parse_price(completion)


specialist.__name__ = "Fine-tuned Qwen2.5-3B"
evaluate(specialist, test, size=250)

In [ ]:
def weighted(wine: Wine, top: int = 8) -> float:
    """Expected price under the model's own distribution over the first answer token."""
    inputs = tokenizer(wine.test_prompt(), return_tensors="pt").to("cuda")
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1]
    probabilities = torch.nn.functional.softmax(logits, dim=-1)
    values, indices = probabilities.topk(top)
    prices, weights = [], []
    for probability, index in zip(values.tolist(), indices.tolist(), strict=True):
        price = parse_price(tokenizer.decode(index))
        if price:
            prices.append(price)
            weights.append(probability)
    if not prices:
        return 0.0
    total = sum(weights)
    return sum(price * weight for price, weight in zip(prices, weights, strict=True)) / total


weighted.__name__ = "Fine-tuned Qwen2.5-3B (weighted)"
evaluate(weighted, test, size=250)

### Experiments

- **Base model, untrained** on the same prompts: the gap is what the fine-tune actually bought.
- **Rank sweep**: r = 8 / 32 / 64 at matched steps.
- **Summaries vs full notes** (`-summaries` dataset from the previous notebook).
- **Add `points` to the prompt** and watch the fine-tune coast -- the same leakage the baselines see.
- **Bigger base**: an 8B model in 4-bit still fits an A100. Does scale beat data curation here?